# DistilBERT — Toxic Comment Classification

This notebook fine-tunes DistilBERT on the Jigsaw dataset for binary toxicity detection.
DistilBERT is a smaller, faster version of BERT that retains ~97% of its performance.
Unlike the NBOW+LR model, DistilBERT captures word context, so it can potentially
handle sarcasm and negation better.

**Prereq:** Run `notebooks/preprocessing.ipynb` first to generate the split CSVs.

In [4]:
import sys
sys.path.insert(0, "..")

import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src.data_utils import load_splits, split_summary, RANDOM_SEED
from src.distilbert_model import (
    train_distilbert,
    predict,
    plot_training_curves,
    save_model,
    get_device,
    MAX_LEN,
)
from src.evaluation import evaluate, extract_errors
from pathlib import Path

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {get_device()}")

ModuleNotFoundError: No module named 'sklearn'

## 1. Load data

We use the same 80/10/10 stratified splits as the other models.

In [ ]:
train, val, test = load_splits(data_dir="../data")
split_summary(train, val, test)

### Token length analysis

We need to pick `MAX_LEN` — the number of tokens we truncate/pad to.
Too short and we lose context; too long and training is slow.
DistilBERT's tokenizer produces more tokens than a whitespace split, so we
check the actual token count distribution here.

In [ ]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# Sample 5000 comments to estimate token length distribution (full dataset is slow)
sample = train.sample(5000, random_state=RANDOM_SEED)["comment_text"].fillna("")
token_lens = sample.apply(lambda t: len(tokenizer.encode(t, add_special_tokens=True)))

print(f"Token length stats (n=5000 sample):")
print(token_lens.describe(percentiles=[0.75, 0.90, 0.95, 0.99]).round(1))
print(f"\nMAX_LEN={MAX_LEN} covers {(token_lens <= MAX_LEN).mean()*100:.1f}% of sampled comments")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(token_lens.clip(upper=300), bins=50, color="#378ADD", alpha=0.8, edgecolor="white")
ax.axvline(MAX_LEN, color="#D85A30", linestyle="--", label=f"MAX_LEN = {MAX_LEN}")
ax.set_xlabel("Token count (clipped at 300)")
ax.set_ylabel("Number of comments")
ax.set_title("Token length distribution (5k sample from train)")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "distilbert_token_lengths.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Fine-tune DistilBERT

Key choices:
- **3 epochs** — standard for BERT fine-tuning; more tends to overfit
- **lr = 2e-5** — the canonical learning rate for BERT fine-tuning
- **Weighted cross-entropy loss** — handles the 9:1 class imbalance
- **Gradient clipping** — prevents exploding gradients when adapting to a new domain
- **Best checkpoint** by validation F1 is automatically saved

Training takes ~15-30 min on a GPU, longer on CPU.

In [ ]:
model, tokenizer, history = train_distilbert(
    train,
    val,
    batch_size=32,
    epochs=3,
    lr=2e-5,
)

In [ ]:
plot_training_curves(history)

## 3. Evaluate on validation set

In [ ]:
val_preds, val_probs = predict(model, tokenizer, val)
val_results = evaluate(
    val["label"].values,
    val_preds,
    model_name="DistilBERT (val)",
    y_prob=val_probs,
)

## 4. Evaluate on test set

In [ ]:
test_preds, test_probs = predict(model, tokenizer, test)
test_results = evaluate(
    test["label"].values,
    test_preds,
    model_name="DistilBERT",
    y_prob=test_probs,
)
print("\nTest results:", test_results)

## 5. Save predictions, results, and model

In [ ]:
# Save predictions CSV (used by error_analysis.ipynb)
test_out = test.copy()
test_out["distilbert_pred"] = test_preds
test_out["distilbert_prob"] = test_probs.round(6)
test_out.to_csv(RESULTS_DIR / "test_distilbert_preds.csv", index=False)
print(f"Predictions saved -> {RESULTS_DIR / 'test_distilbert_preds.csv'}")

# Save results JSON (used by evaluation comparison notebook)
test_results["epochs"]     = 3
test_results["lr"]         = 2e-5
test_results["batch_size"] = 32
test_results["history"]    = history
with open(RESULTS_DIR / "distilbert_results.json", "w") as f:
    json.dump(test_results, f, indent=2)
print(f"Results saved     -> {RESULTS_DIR / 'distilbert_results.json'}")

In [ ]:
# Save model weights
save_model(model, tokenizer, models_dir="../models")

In [ ]:
# Save false positive / false negative samples for error analysis
errors = extract_errors(test_out, pred_col="distilbert_pred")
errors["false_positives"].to_csv(RESULTS_DIR / "errors_distilbert_fp.csv", index=False)
errors["false_negatives"].to_csv(RESULTS_DIR / "errors_distilbert_fn.csv", index=False)
print(f"Error samples saved -> {RESULTS_DIR}/errors_distilbert_*.csv")

## 6. Quick look at errors

Spot-check some false positives and false negatives to get an intuition for
where the model struggles. Full error analysis is in `error_analysis.ipynb`.

In [ ]:
fp = errors["false_positives"]
fn = errors["false_negatives"]

print("=== False Positives (predicted toxic, actually non-toxic) ===")
for _, row in fp.head(5).iterrows():
    prob = test_out.loc[row.name, "distilbert_prob"]
    print(f"  prob={prob:.3f}  |  {str(row['comment_text'])[:200]}")
    print()

print("=== False Negatives (missed toxic comments) ===")
for _, row in fn.head(5).iterrows():
    prob = test_out.loc[row.name, "distilbert_prob"]
    print(f"  prob={prob:.3f}  |  {str(row['comment_text'])[:200]}")
    print()